# Exploratory Data Analysis

## 1. Import libraries

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import plotly.express.colors as pc
import plotly.graph_objects as go
from plotly.subplots import make_subplots



## 2. Load metadata

In [9]:
df = pd.read_csv("fragments_metadata.csv")
df

,id,name,age,gender,position,record_id,segment,label,category,duration
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525
4,5,P2,5.3,0,p4,25284,0,Fine Crackle,Adventitious,1.93425
...,...,...,...,...,...,...,...,...,...,...
24573,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025
24574,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525
24575,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525
24576,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925


## 3. EDA

### 3.1. Label distribution

#### Results
_To be filled after running the notebook._

In [10]:
# --- 1. Preparar datos ---
total_n = len(df)

counts = df.groupby(["category", "label"]).size().reset_index(name="count")
counts["proportion"] = counts["count"] / total_n

adv_counts = (
    df[df["label"] != "Normal"]["label"]
    .value_counts(normalize=True)
    .rename_axis("label")
    .reset_index(name="proportion")
)

adv_order = adv_counts.sort_values("proportion", ascending=False)["label"].tolist()
labels_order = ["Normal"] + adv_order

# --- 2. Paleta temática: verde-azulado para Normal, gradiente cálido (rojo->amarillo claro)
#     para adventicios, del más frecuente (más intenso) al menos frecuente (más claro) ---
n_adv = len(adv_order)
adv_colors = pc.sample_colorscale(
    "OrRd", [0.95 - 0.65 * (i / max(n_adv - 1, 1)) for i in range(n_adv)]
)
color_map = {"Normal": "#2CA58D"}
color_map.update({lab: c for lab, c in zip(adv_order, adv_colors)})

# --- 3. Crear figura ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Normal vs Adventitious", "Adventitious type distribution"),
    specs=[[{"type": "bar"}, {"type": "domain"}]],
)

# --- Subplot 1: barra apilada. Solo mostramos texto si el segmento es >= 4% ---
TEXT_THRESHOLD = 0.04
for lab in labels_order:
    sub = counts[counts["label"] == lab]
    if sub.empty:
        continue
    texts = sub["proportion"].map(lambda p: f"{p:.1%}" if p >= TEXT_THRESHOLD else "")
    fig.add_trace(
        go.Bar(
            x=sub["category"],
            y=sub["proportion"],
            name=lab,
            marker_color=color_map[lab],
            legendgroup=lab,
            text=texts,
            textposition="inside",
            textfont=dict(size=13, color="white"),
            hovertemplate=f"{lab}: %{{y:.1%}}<extra></extra>",
        ),
        row=1, col=1
    )

# --- Subplot 2: pie, sin labels, solo % con posicionamiento automático (evita solapes) ---
fig.add_trace(
    go.Pie(
        labels=adv_counts["label"],
        values=adv_counts["proportion"],
        name="Adventitious",
        marker_colors=[color_map[lab] for lab in adv_counts["label"]],
        textinfo="percent",
        textposition="auto",
        showlegend=False,
        sort=False,  # respeta el orden de más a menos que ya trae adv_counts... 
    ),
    row=1, col=2
)
# aseguramos orden más->menos en el pie también
fig.data[-1].labels = adv_order
fig.data[-1].values = [adv_counts.set_index("label").loc[lab, "proportion"] for lab in adv_order]
fig.data[-1].marker.colors = [color_map[lab] for lab in adv_order]

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    barmode="stack",
    title_text="Label distribution",
    showlegend=True,
    legend=dict(traceorder="normal", font=dict(size=14)),
    font=dict(size=15),
    title_font=dict(size=20),
    plot_bgcolor="white",
)
fig.update_annotations(font_size=16)

fig.update_xaxes(
    title_text="Category",
    categoryorder="array",
    categoryarray=["Normal", "Adventitious"],
    row=1, col=1
)
fig.update_yaxes(title_text="Proportion", tickformat=".0%", row=1, col=1)

fig.show()

### 3.2. Duration distribution

#### Results
_To be filled after running the notebook._

In [35]:
# --- 1. Datos ---
durations = df["duration"].dropna()

# --- 2. Estimar KDE ---
kde = gaussian_kde(durations)
x_kde = np.linspace(durations.min(), durations.max(), 300)
y_kde = kde(x_kde)

# --- 3. Crear figura: 2 filas, eje X compartido ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],   # histograma más grande, boxplot como "resumen" fino
    vertical_spacing=0.03,
)

# --- Histograma (normalizado a densidad para que la escala coincida con la KDE) ---
fig.add_trace(
    go.Histogram(
        x=durations,
        histnorm="probability density",
        name="Duration",
        marker_color="chocolate",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

# --- KDE ---
fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="saddlebrown", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

# --- Boxplot horizontal ---
fig.add_trace(
    go.Box(
        x=durations,
        name="Duration",
        marker_color="chocolate",
        line_color="chocolate",
        boxpoints="outliers",   # muestra outliers como puntos
        showlegend=False,
        boxmean="sd"
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Duration distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Duration (s)", row=2, col=1)

fig.show()

In [28]:
df["duration"].describe()

count    24578.000000
mean         1.739736
std          0.753394
min          0.126250
25%          1.208250
50%          1.682250
75%          2.184250
max          9.273250
Name: duration, dtype: float64

### 3.3. Number of fragments per patient

#### Results
_To be filled after running the notebook._

In [57]:
# --- 1. Contar fragmentos por paciente ---
fragments_per_patient = df.groupby("name").size()

# --- 2. Estimar KDE (si hay suficiente variación en los valores) ---
kde = gaussian_kde(fragments_per_patient)
x_kde = np.linspace(fragments_per_patient.min(), fragments_per_patient.max(), 300)
y_kde = kde(x_kde)

# --- 3. Figura: histograma + KDE arriba, boxplot horizontal abajo ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

fig.add_trace(
    go.Histogram(
        x=fragments_per_patient,
        histnorm="probability density",
        name="n/patient",
        marker_color="mediumpurple",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="rebeccapurple", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        x=fragments_per_patient,
        name="Fragments/patient",
        marker_color="mediumpurple",
        line_color="mediumpurple",
        boxpoints="outliers",
        showlegend=False,
        boxmean="sd",
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Number of fragments per patient distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Number of fragments per patient", row=2, col=1)

fig.show()

In [62]:
fragments_per_patient.describe()

count    958.000000
mean      25.655532
std       30.165623
min        1.000000
25%        8.000000
50%       17.000000
75%       32.000000
max      298.000000
dtype: float64

### 3.4. Labels per patient distribution

#### Results
_To be filled after running the notebook._

In [64]:
# --- 1. Número de labels distintos por paciente ---
labels_per_patient = df.groupby("name")["label"].nunique()

# --- 3. Figura: histograma + KDE arriba, boxplot horizontal abajo ---
fig = make_subplots(
    rows=1, cols=1,
    shared_xaxes=True,
    row_heights=[1],
    vertical_spacing=0.03,
)

fig.add_trace(
    go.Histogram(
        x=labels_per_patient,
        name="Labels/paciente",
        marker_color="indianred",
        opacity=0.75,
        xbins=dict(size=1),
        showlegend=False,
    ),
    row=1, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Number of labels per patient distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Number of different labels")

fig.show()

In [63]:
labels_per_patient.describe()

count    958.000000
mean       1.460334
std        0.791803
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        6.000000
Name: label, dtype: float64